In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import sqlite3
import sys
from tqdm import tqdm

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.getcwd()))
from DeepUnitMatch.testing.fast_testing import AUC
from DeepUnitMatch.testing.test import remove_conflicts
from DeepUnitMatch.utils.helpers import pick, avg_across_directions

sqlite_db_path = "/path/to/matchtables.db"
metadata_path = "path/to/metadata_index.json"

DUM_col = "NBProb18mice"
UM_col = "NBProb18mice_NoSpat"
distance_col = "CentroidDist"
functional_col = "newISI"

In [ ]:
conn = sqlite3.connect(sqlite_db_path)
mt = pd.read_sql_query(
    f"SELECT ID1,ID2,RecSes1,RecSes2,{UM_col},{DUM_col},{functional_col},{distance_col} FROM AL032_19011111882_2",
    conn,
)

df = mt.loc[mt["RecSes1"] != mt["RecSes2"]]

In [ ]:
import importlib 
import DeepUnitMatch.utils.helpers
importlib.reload(DeepUnitMatch.utils.helpers)
from DeepUnitMatch.utils.helpers import avg_across_directions, pick

In [ ]:
# Figure 2e

def AUCvsNmatches(mt, column, um_col, func_metric, r1, r2, vis=False):

    mt = pick(mt, r1, r2, False)
    mt = avg_across_directions(mt, columns=[column, um_col, func_metric])

    bycol, col_dropped = remove_conflicts(mt, column)
    bymp, mp_dropped = remove_conflicts(mt, um_col)

    bycol = bycol.sort_values(by=column, ascending=False)
    bymp = bymp.sort_values(by=um_col, ascending=False)

    bycol = bycol.loc[bycol["RecSes1"] < bycol["RecSes2"]]
    bymp = bymp.loc[bymp["RecSes1"] < bymp["RecSes2"]]

    filtered_N = len(mt) // 2

    x1, x2 = [], []
    y1, y2 = [], []
    m1, m2 = True, True

    for i in np.arange(5, filtered_N, step=5)[1:]:
        i = int(i)
        d_matches = bycol.head(i)
        u_matches = bymp.head(i)
        if len(d_matches) < i:
            m1 = False
        if len(u_matches) < i:
            m2 = False
        if m1 or m2:
            auc1 = AUC(mt, d_matches.index, func_metric)
            auc2 = AUC(mt, u_matches.index, func_metric)

        if m1:
            x1.append(len(d_matches))
            y1.append(auc1)
        if m2:
            x2.append(len(u_matches))
            y2.append(auc2)

    if vis:
        plt.plot(x1, y1, "o-", markersize=8, color="red", linewidth=2, label="DUM")
        plt.plot(x2, y2, "o-", markersize=8, color="blue", linewidth=2, label="UM")
        plt.xlabel("Number of matches", fontsize=14)
        plt.ylabel("AUC", fontsize=14)
        plt.xticks(np.arange(0, 170, 25))
        plt.yticks(ticks=np.arange(0.82, 0.98, 0.04))

        plt.grid(False)
        plt.tight_layout()

        plt.rcParams["svg.fonttype"] = "none"
        ax = plt.gca()
        ax.spines[["right", "top"]].set_visible(False)
        ax.spines["left"].set_position(("outward", 15))
        ax.spines["bottom"].set_position(("outward", 15))
        plt.show()
    return x1, x2, y1, y2


# Single session pair
AUCvsNmatches(df, DUM_col, UM_col, functional_col, 12, 13, vis=True)

In [ ]:
# Figure 2f


def relativeAUCvsN(mt, col, um_col, func, consec_only=False, vis=True, cutoff=None):

    sessions = mt["RecSes1"].unique()

    N, relAUC, baseline = [], [], []
    max_len = 0

    # Collect AUCs
    for r1 in sessions:
        for r2 in sessions:
            if r1 >= r2:
                continue
            if consec_only and abs(r2 - r1) > 1:
                continue

            dm_x, um_x, dm_y, um_y = AUCvsNmatches(mt, col, um_col, func, r1, r2)

            x = min(len(dm_x), len(um_x))
            if x > max_len:
                max_len = x
            N.append(dm_x[:x])
            dm_y, um_y = np.array(dm_y[:x]), np.array(um_y[:x])
            relAUC.append(dm_y - um_y)
            baseline.append(um_y - um_y)

    # Get average
    average_diff = []
    for i in range(max_len):
        vals = []
        for l in relAUC:
            if i < len(l):
                vals.append(l[i])
        average_diff.append(np.mean(vals))

    # Plot data
    if vis:
        not_plotted = True
        for i, (x, y1, y2) in enumerate(zip(N, relAUC, baseline)):
            if len(x) == max_len and not_plotted:
                plt.plot(
                    x,
                    y1,
                    alpha=0.3,
                    color="red",
                    linewidth=1,
                    label="DUM relative to UM",
                )
                plt.plot(x, y2, color="blue", linewidth=2, label="UM", zorder=100)
                plt.plot(
                    x,
                    average_diff,
                    "o-",
                    markersize=2,
                    color="black",
                    linewidth=2,
                    label="Average difference",
                    zorder=200,
                )
                not_plotted = False
            else:
                plt.plot(x, y1, alpha=0.3, color="red", linewidth=1)
        plt.xlabel("Number of matches")
        plt.ylabel("AUC difference")
        plt.grid(False)
        plt.xticks(np.arange(0, 150, 25))
        plt.yticks(np.arange(-0.2, 0.4, 0.2))
        if cutoff:
            plt.xlim(0, cutoff)

        plt.legend()
        plt.savefig(os.path.join(dir, "AL032_relAUC.png"), dpi=300)
        plt.rcParams["svg.fonttype"] = "none"
        ax = plt.gca()
        ax.spines[["right", "top"]].set_visible(False)
        ax.spines["left"].set_position(("outward", 15))
        ax.spines["bottom"].set_position(("outward", 15))
        plt.show()

    return average_diff


avg_line = relativeAUCvsN(mt, DUM_col, UM_col, functional_col, consec_only=False, cutoff=150)

In [ ]:
# Figure 2g - this is slow

lines = []
mice = []
longest_length = 0

metadata = json.load(open(metadata_path))

for entry in tqdm(metadata, desc="Processing mice", total=len(metadata)):
    mouse, probe, loc = entry["mouse"], entry["probe"], entry["loc"]
    try:
        mt = pd.read_sql_query(
            f"SELECT ID1,ID2,RecSes1,RecSes2,{DUM_col},{UM_col},{functional_col},{distance_col} FROM {mouse}_{probe}_{loc}",
            conn,
        )
    except Exception as e:
        print("Failed to read match table for ", mouse, probe, loc)
        print("Reason: ", e)
        continue
    df = mt.loc[mt["RecSes1"] != mt["RecSes2"]]
    avg_diff = relativeAUCvsN(df, DUM_col, UM_col, functional_col, vis=False)
    lines.append(avg_diff)
    mice.append(mouse)
    if len(avg_diff) > longest_length:
        longest_length = len(avg_diff)

X = []
avg_line = []
for i in range(longest_length):
    X.append(10 + i * 5)
    vals = []
    for l in lines:
        if i < len(l):
            vals.append(l[i])
    avg_line.append(np.mean(vals))

seen_label, unseen_label = False, False

lines = [l for l in lines if len(l) > 0]

for line in lines:
    if len(line) == longest_length:
        plt.plot(
            X, [0] * longest_length, color="blue", linewidth=2, label="UM", zorder=100
        )
        plt.plot(
            X,
            avg_line,
            color="black",
            linewidth=2,
            label="Average over mice",
            zorder=200,
        )
        plt.plot(
            X, line, alpha=0.3, color="red", linewidth=1, label="DUM relative to UM"
        )
    elif max(X[: len(line)]) <= 30:
        # Skip the very poor sets of recording sessions
        continue
    else:
        plt.plot(X[: len(line)], line, alpha=0.3, color="red", linewidth=1)

plt.xlabel("Number of matches")
plt.ylabel("AUC difference")
plt.xlim(0, 150)
plt.yticks(np.arange(-0.1, 0.2, 0.1))
plt.grid(False)
plt.rcParams["svg.fonttype"] = "none"
ax = plt.gca()
ax.spines[["right", "top"]].set_visible(False)
ax.spines["left"].set_position(("outward", 15))
ax.spines["bottom"].set_position(("outward", 15))
plt.show()